 # Language Translator UI

### 1. Set up the UI for text input and language selection

In [1]:
import ipywidgets as widgets
from IPython.display import display

# Text input widget
text_input = widgets.Textarea(
    value='',
    placeholder='Type your text here...',
    description='Text to translate:',
    disabled=False,
    layout={'height': '150px', 'width': '80%'}
)

# Language selection widget
# For simplicity, let's include a few common languages.
# You can expand this list as needed.
language_options = {
    'English': 'en',
    'Spanish': 'es',
    'French': 'fr',
    'German': 'de',
    'Italian': 'it',
    'Japanese': 'ja',
    'Korean': 'ko',
    'Chinese (Simplified)': 'zh-CN'
}

language_selector = widgets.Dropdown(
    options=language_options,
    value='es', # Default to Spanish
    description='Translate to:',
    disabled=False,
)

# Output widget to display translated text
translated_output = widgets.Output()

display(text_input, language_selector, translated_output)

Textarea(value='', description='Text to translate:', layout=Layout(height='150px', width='80%'), placeholder='…

Dropdown(description='Translate to:', index=1, options={'English': 'en', 'Spanish': 'es', 'French': 'fr', 'Ger…

Output()

### 2. Integrate Translation API

For translation, we'll use the `googletrans` library, which is a free and unlimited Python library that implements the Google Translate API. We'll need to install it first.

In [2]:
!pip install googletrans==4.0.0-rc1

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.1/55.1 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 4.2 MB/s eta 0:00:00
  Created wheel for googletrans: filename=googletrans-4.0.0rc1-py3-none-any.whl size=17396 sha256=686b6b662a763b8fb739e9b9ecc19f49432d96bcc464238833d2ffef307f72ed
  Stored in directory: /root/.cache/pip/wheels/95/0f/04/b17a72024b56a60e499ce1a6313d283ed5ba332407155bee03
Successfully built googletrans
  Attempting uninstall: hyperframe
    Found existing installation: hyperframe 6.1.0
    Uninstalling hyperfram

In [3]:
from googletrans import Translator

def translate_text(text, dest_lang):
    translator = Translator()
    try:
        translation = translator.translate(text, dest=dest_lang)
        return translation.text
    except Exception as e:
        return f"Translation error: {e}"

def on_translate_button_click(b):
    with translated_output:
        translated_output.clear_output()
        if text_input.value:
            translated_text = translate_text(text_input.value, language_selector.value)
            print(f"Translated Text: {translated_text}")
        else:
            print("Please enter some text to translate.")

# Create a translate button
translate_button = widgets.Button(description="Translate")
translate_button.on_click(on_translate_button_click)

display(translate_button)

Button(description='Translate', style=ButtonStyle())

### 3. Add Text-to-Speech (Optional)

In [6]:
!pip install gTTS

In [7]:
from gtts import gTTS
from IPython.display import Audio, display

def text_to_speech(text, lang):
    try:
        tts = gTTS(text=text, lang=lang, slow=False)
        # Save the audio to a BytesIO object
        from io import BytesIO
        fp = BytesIO()
        tts.write_to_fp(fp)
        fp.seek(0)
        return Audio(fp.read(), autoplay=True)
    except Exception as e:
        print(f"Text-to-speech error: {e}")
        return None

def on_speak_button_click(b):
    with translated_output:
        if text_input.value and hasattr(on_translate_button_click, 'translated_text_value'):
            # Use the most recently translated text
            spoken_audio = text_to_speech(on_translate_button_click.translated_text_value, language_selector.value)
            if spoken_audio:
                display(spoken_audio)
        else:
            print("Please translate some text first.")

# Modify the on_translate_button_click to store the translated text
def on_translate_button_click_modified(b):
    with translated_output:
        translated_output.clear_output()
        if text_input.value:
            translated_text = translate_text(text_input.value, language_selector.value)
            print(f"Translated Text: {translated_text}")
            on_translate_button_click.translated_text_value = translated_text # Store for TTS
        else:
            print("Please enter some text to translate.")

# Re-assign the click handler to the modified function
translate_button.on_click(on_translate_button_click_modified)

# Create a speak button
speak_button = widgets.Button(description="Speak Translated Text")
speak_button.on_click(on_speak_button_click)

display(speak_button)

Button(description='Speak Translated Text', style=ButtonStyle())

### 4. Add Copy Feature (Optional)

In [10]:
from IPython.display import display, Javascript

def copy_to_clipboard(text):
    js = Javascript(f'''
        navigator.clipboard.writeText('{text.replace("'", "\\'")}')
        .then(function() {{
            console.log("Copy successful!");
        }})
        .catch(function(err) {{
            console.error("Copy failed: ", err);
        }});
    ''')
    display(js)

def on_copy_button_click(b):
    with translated_output:
        if hasattr(on_translate_button_click, 'translated_text_value') and on_translate_button_click.translated_text_value:
            copy_to_clipboard(on_translate_button_click.translated_text_value)
            print("Translated text copied to clipboard!")
        else:
            print("No text to copy. Please translate some text first.")

# Create a copy button
copy_button = widgets.Button(description="Copy Translated Text")
copy_button.on_click(on_copy_button_click)

display(copy_button)

Button(description='Copy Translated Text', style=ButtonStyle())

### 4. Add Copy Feature (Optional)

In [9]:
from IPython.display import display, Javascript

def copy_to_clipboard(text):
    js = Javascript(f'''
        navigator.clipboard.writeText('{text.replace("'", "\\'")}')
        .then(function() {{
            console.log("Copy successful!");
        }})
        .catch(function(err) {{
            console.error("Copy failed: ", err);
        }});
    ''')
    display(js)

def on_copy_button_click(b):
    with translated_output:
        if hasattr(on_translate_button_click, 'translated_text_value') and on_translate_button_click.translated_text_value:
            copy_to_clipboard(on_translate_button_click.translated_text_value)
            print("Translated text copied to clipboard!")
        else:
            print("No text to copy. Please translate some text first.")

# Create a copy button
copy_button = widgets.Button(description="Copy Translated Text")
copy_button.on_click(on_copy_button_click)

display(copy_button)

Button(description='Copy Translated Text', style=ButtonStyle())

### 4. Add Copy Feature (Optional)

In [8]:
from IPython.display import display, Javascript

def copy_to_clipboard(text):
    js = Javascript(f'''
        navigator.clipboard.writeText('{text.replace("'", "\\'")}')
        .then(function() {{
            console.log("Copy successful!");
        }})
        .catch(function(err) {{
            console.error("Copy failed: ", err);
        }});
    ''')
    display(js)

def on_copy_button_click(b):
    with translated_output:
        if hasattr(on_translate_button_click, 'translated_text_value') and on_translate_button_click.translated_text_value:
            copy_to_clipboard(on_translate_button_click.translated_text_value)
            print("Translated text copied to clipboard!")
        else:
            print("No text to copy. Please translate some text first.")

# Create a copy button
copy_button = widgets.Button(description="Copy Translated Text")
copy_button.on_click(on_copy_button_click)

display(copy_button)

Button(description='Copy Translated Text', style=ButtonStyle())

### 3. Add Text-to-Speech (Optional)

In [4]:
!pip install gTTS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 3.7 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.4.2
    Uninstalling click-8.4.2:
      Successfully uninstalled click-8.4.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.25.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.
google-adk 2.3.0 requires httpx<1,>=0.27, but you have httpx 0.13.3 which is incompatible.
gradio 6.19.0 requires httpx<1.0,>=0.24.1, but you have httpx 0.13.3 which is incompatible.
huggingface-hub 1.20.1 requires click>=8.4.0, but you have click 8.1.8 which is incompatible.
huggingface-hub 1.20.1 requires httpx<1,>=0.23.0, but you have httpx 0.13.3 which is incompatible.
weasel 1.0.0 requires httpx>=0.24.0, but you have httpx 0.13.3 which is incompatible.
gradio-client 2.5.0 requires httpx>=0.24.1, but you h

In [5]:
from gtts import gTTS
from IPython.display import Audio, display

def text_to_speech(text, lang):
    try:
        tts = gTTS(text=text, lang=lang, slow=False)
        # Save the audio to a BytesIO object
        from io import BytesIO
        fp = BytesIO()
        tts.write_to_fp(fp)
        fp.seek(0)
        return Audio(fp.read(), autoplay=True)
    except Exception as e:
        print(f"Text-to-speech error: {e}")
        return None

def on_speak_button_click(b):
    with translated_output:
        if text_input.value and hasattr(on_translate_button_click, 'translated_text_value'):
            # Use the most recently translated text
            spoken_audio = text_to_speech(on_translate_button_click.translated_text_value, language_selector.value)
            if spoken_audio:
                display(spoken_audio)
        else:
            print("Please translate some text first.")

# Modify the on_translate_button_click to store the translated text
def on_translate_button_click_modified(b):
    with translated_output:
        translated_output.clear_output()
        if text_input.value:
            translated_text = translate_text(text_input.value, language_selector.value)
            print(f"Translated Text: {translated_text}")
            on_translate_button_click.translated_text_value = translated_text # Store for TTS
        else:
            print("Please enter some text to translate.")

# Re-assign the click handler to the modified function
translate_button.on_click(on_translate_button_click_modified)

# Create a speak button
speak_button = widgets.Button(description="Speak Translated Text")
speak_button.on_click(on_speak_button_click)

display(speak_button)

Button(description='Speak Translated Text', style=ButtonStyle())